# OpenVoice V2 — Клонирование голоса (русский язык)

**Что делает этот ноутбук:**
- Клонирует голос по 3-10 секундам аудио
- Сохраняет картавость и другие особенности
- Поддерживает русский язык
- Работает бесплатно на Google Colab GPU

**Шаги:**
1. Запусти все ячейки по порядку (Shift+Enter)
2. В шаге 4 загрузи свой аудиофайл (WAV или MP3)
3. В шаге 5 введи текст, который хочешь озвучить
4. В шаге 6 скачай результат


## Шаг 1: Проверь, что GPU включён
В меню: **Среда выполнения → Сменить среду выполнения → T4 GPU → Сохранить**

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('GPU найден!')
    print(result.stdout[:500])
else:
    print('GPU НЕ найден. Перейди: Среда выполнения → Сменить среду → T4 GPU')

## Шаг 2: Установка OpenVoice V2
Займёт 2-3 минуты

In [ ]:
# Клонируем репозиторий
!git clone https://github.com/myshell-ai/OpenVoice.git
%cd OpenVoice

# Устанавливаем зависимости
!pip install -q -e .
!pip install -q faster-whisper
!pip install -q whisper-timestamped
!pip install -q transformers==4.38.0

print('\nУстановка завершена!')

## Шаг 3: Загрузка модели

In [ ]:
import os
os.chdir('/content/OpenVoice')

# Скачиваем веса модели
!git lfs install
!huggingface-cli download myshell-ai/OpenVoiceV2 --local-dir checkpoints_v2

print('Модель загружена!')

In [ ]:
import torch
from openvoice import se_extractor
from openvoice.api import ToneColorConverter
from melo.api import TTS
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Используем: {device}')

# Загружаем конвертер тон-колора
ckpt_converter = 'checkpoints_v2/converter'
tone_color_converter = ToneColorConverter(f'{ckpt_converter}/config.json', device=device)
tone_color_converter.load_ckpt(f'{ckpt_converter}/checkpoint.pth')

print('Модель готова к работе!')

## Шаг 4: Загрузи свой голос

**Требования к аудиофайлу:**
- Формат: WAV или MP3
- Длина: от 3 до 30 секунд (оптимально 10-15 сек)
- Качество: без фоновой музыки, без эха
- В записи должен быть слышен картавый звук [р]

**Рекомендуемый текст для записи эталона:**
> "Рыжая лиса прыгнула через реку. Красные розы растут в ряду. Прораб проверил работу рабочих."

In [ ]:
# Загружаем файл через интерфейс Colab
from google.colab import files
import shutil

print('Выбери свой аудиофайл (WAV или MP3)...')
uploaded = files.upload()

# Сохраняем загруженный файл
for filename in uploaded.keys():
    reference_audio = f'/content/reference_voice.wav'
    with open(reference_audio, 'wb') as f:
        f.write(uploaded[filename])
    print(f'Файл загружен: {filename}')
    print(f'Сохранён как: {reference_audio}')

In [ ]:
# Извлекаем характеристики голоса из твоего аудио
import os
os.makedirs('outputs_v2', exist_ok=True)

reference_speaker = reference_audio
target_se, audio_name = se_extractor.get_se(reference_speaker, tone_color_converter, vad=False)

print('Характеристики голоса извлечены успешно!')
print('Теперь можно синтезировать речь с твоим голосом.')

## Шаг 5: Введи текст для озвучки

In [ ]:
# ИЗМЕНИ ЭТОТ ТЕКСТ НА СВОЙ
text_to_speak = """Привет! Это клон моего голоса. 
Обратите внимание, что мой голос сохранил все особенности произношения.
Рыжая лиса прыгнула через реку. Красные розы растут в ряду."""

# Скорость речи (1.0 = нормальная, 0.9 = чуть медленнее)
speed = 1.0

print(f'Текст для синтеза ({len(text_to_speak)} символов):')
print(text_to_speak)

In [ ]:
# Синтез речи и клонирование голоса
import IPython.display as ipd

output_dir = 'outputs_v2'

# Загружаем базовую TTS модель для русского языка
tts_model = TTS(language='RU', device=device)
speaker_ids = tts_model.hps.data.spk2id
print('Доступные голоса:', list(speaker_ids.keys()))

# Используем первый доступный голос как основу
src_path = f'{output_dir}/tmp_base.wav'
tts_model.tts_to_file(text_to_speak, list(speaker_ids.values())[0], src_path, speed=speed)
print('Базовый голос синтезирован.')

# Применяем характеристики твоего голоса
save_path = f'{output_dir}/output_cloned_voice.wav'

encode_message = "@MyShell"
tone_color_converter.convert(
    audio_src_path=src_path,
    src_se=None,
    tgt_se=target_se,
    output_path=save_path,
    message=encode_message
)

print(f'Готово! Файл сохранён: {save_path}')
print('Воспроизводим результат...')
ipd.display(ipd.Audio(save_path, rate=22050))

## Шаг 6: Скачай результат

In [ ]:
from google.colab import files
files.download(save_path)
print('Файл скачан!')

## Дополнительно: Попробуй разные варианты
Можно менять скорость и запускать снова

In [ ]:
# Пробуем разные параметры
variants = [
    {'speed': 0.85, 'name': 'slow'},
    {'speed': 1.0,  'name': 'normal'},
    {'speed': 1.15, 'name': 'fast'},
]

short_text = "Рыжая лиса прыгнула через реку. Красные розы растут в ряду."

for v in variants:
    base = f'{output_dir}/tmp_{v["name"]}.wav'
    out = f'{output_dir}/variant_{v["name"]}.wav'
    
    tts_model.tts_to_file(short_text, list(speaker_ids.values())[0], base, speed=v['speed'])
    tone_color_converter.convert(
        audio_src_path=base,
        src_se=None,
        tgt_se=target_se,
        output_path=out,
        message="@MyShell"
    )
    print(f'\nВариант: {v["name"]} (скорость {v["speed"]})')
    ipd.display(ipd.Audio(out, rate=22050))